In [12]:
import requests
from bs4 import BeautifulSoup
import os
from urllib.parse import urljoin
from IPython.display import Image, display  

# 1. 뉴스 URL
news_url = 'https://news.nate.com/'  
res = requests.get(news_url)
res.encoding = 'utf-8'
print(f"응답 코드: {res.status_code}")

# 2. HTML 파싱
if res.ok:
    soup = BeautifulSoup(res.text, 'html.parser')

    # 뉴스 블록 선택
    news_list = soup.select("div.postList > ul > li")

    # 이미지 저장 폴더 생성
    img_folder = 'img'
    if not os.path.isdir(img_folder):
        os.mkdir(img_folder)

    # 3. 뉴스 반복하며 데이터 추출
    for news in news_list[:5]:  # 5개만 예시로
        # 제목과 링크
        a_tag = news.select_one("a.thumb")
        if not a_tag:
            continue

        title_tag = news.select_one("strong")
        title = title_tag.text.strip() if title_tag else "제목 없음"
        link = urljoin("https://news.nate.com", a_tag["href"])

        print(f"\n 제목: {title}")
        print(f" 링크: {link}")

        # 이미지 처리
        img_tag = news.select_one("img")
        if img_tag and 'src' in img_tag.attrs:
            src = img_tag["src"].strip()
            if src.startswith("//"):
                src = "https:" + src
            else:
                src = urljoin("https://news.nate.com", src)

            # 이미지 다운로드
            img_res = requests.get(src)
            if img_res.ok:
                img_data = img_res.content
                file_name = os.path.join(img_folder, os.path.basename(src))
                with open(file_name, 'wb') as f:
                    f.write(img_data)
                    print(f" 이미지 저장: {file_name} ({len(img_data):,} bytes)")

                #  이미지 화면에 출력
                display(Image(data=img_data))
            else:
                print(f" 이미지 요청 실패: {img_res.status_code}")
        else:
            print(" [이미지 없음]")

응답 코드: 200


In [7]:
print()

In [10]:
def download_one_episode(title, no, url):
    import requests
    from bs4 import BeautifulSoup
    import os

    # referer 헤더 설정
    req_header = {
        'referer': url
    }

    # 요청
    res = requests.get(url, headers=req_header)
    print(f"[응답 성공 여부] {res.ok}")
    
    if res.ok:
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # 본문 이미지 URL만 추출
        imgurl_list = []
        img_tags = soup.select("img[src*='IMAG']")  # 웹툰 본문 이미지
        print(f"[이미지 수] {len(img_tags)}")
        
        for img_tag in img_tags:
            imgurl_list.append(img_tag['src'])

        # 디렉토리 생성: img/제목/회차번호/
        dir_name = os.path.join('img', title, str(no))
        print(f"[저장 경로] {dir_name}")
        if not os.path.isdir(dir_name):
            os.makedirs(dir_name)

        # 이미지 다운로드
        for idx, img_url in enumerate(imgurl_list, 1):
            print(f"{idx}: {img_url}")
            res = requests.get(img_url, headers=req_header)
            if res.ok:
                img_data = res.content
                file_name = f"{idx}.jpg"
                file_path = os.path.join(dir_name, file_name)

                with open(file_path, 'wb') as file:
                    file.write(img_data)
                    print(f" 저장 완료: {file_path} (크기: {len(img_data):,} bytes)")
            else:
                print(f" 다운로드 실패: {img_url}")

download_one_episode(
    '용사가 돌아왔다',
    132,
    'https://comic.naver.com/webtoon/detail?titleId=797443&no=132&week=wed'
)

[응답 성공 여부] True
[이미지 수] 79
[저장 경로] img\용사가 돌아왔다\132
1: https://image-comic.pstatic.net/webtoon/797443/132/20250617105532_8d484017aaa2404e00a0b9786e9917b8_IMAG01_1.jpg
 저장 완료: img\용사가 돌아왔다\132\1.jpg (크기: 101,031 bytes)
2: https://image-comic.pstatic.net/webtoon/797443/132/20250617105532_8d484017aaa2404e00a0b9786e9917b8_IMAG01_2.jpg
 저장 완료: img\용사가 돌아왔다\132\2.jpg (크기: 133,927 bytes)
3: https://image-comic.pstatic.net/webtoon/797443/132/20250617105532_8d484017aaa2404e00a0b9786e9917b8_IMAG01_3.jpg
 저장 완료: img\용사가 돌아왔다\132\3.jpg (크기: 125,997 bytes)
4: https://image-comic.pstatic.net/webtoon/797443/132/20250617105532_8d484017aaa2404e00a0b9786e9917b8_IMAG01_4.jpg
 저장 완료: img\용사가 돌아왔다\132\4.jpg (크기: 133,348 bytes)
5: https://image-comic.pstatic.net/webtoon/797443/132/20250617105532_8d484017aaa2404e00a0b9786e9917b8_IMAG01_5.jpg
 저장 완료: img\용사가 돌아왔다\132\5.jpg (크기: 135,827 bytes)
6: https://image-comic.pstatic.net/webtoon/797443/132/20250617105532_8d484017aaa2404e00a0b9786e9917b8_IMAG01_6.jpg
 저장